# Exploratory Data Analysis
Loading the `dataset.csv` and inspecting the distributions of numerical features for healthy (`0`) and suspicious (`1`) packages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")

# Load dataset
df = pd.read_csv('../data/processed/dataset.csv')
print(f"Dataset shape: {df.shape}")
display(df.head())


## Feature Distributions
We will plot histograms for every numerical feature, split by the target `label` (`0` = Healthy, `1` = Suspicious).

In [ ]:
# Identify numerical features
# Excluding 'label' from the list
numerical_features = [col for col in df.columns if col not in ['name', 'label'] and pd.api.types.is_numeric_dtype(df[col])]

print(f"Found {len(numerical_features)} numerical features.")

# Plot histograms split by class
for feature in numerical_features:
    plt.figure(figsize=(10, 5))
    
    # We use a log scale on x-axis or specific binning if data is heavily skewed
    # For general purpose, we'll try standard histplot, falling back to log scale if range is huge.
    
    unique_vals = df[feature].nunique()
    if unique_vals > 50 and df[feature].max() > 1000:
        # Features like stargazers_count or forks_count are heavily skewed, use log scale
        sns.histplot(data=df, x=feature, hue="label", bins=50, kde=True, log_scale=(True, False))
        plt.title(f"Distribution of {feature} (Log Scale) by Label")
    else:
        sns.histplot(data=df, x=feature, hue="label", bins=50, kde=True)
        plt.title(f"Distribution of {feature} by Label")
        
    plt.tight_layout()
    plt.show()


## Insights and Observations

### Top 3 Most Separating Features
Based on visual inspection and preliminary statistical analysis (e.g. AUC), the distributions of these features differ most significantly between healthy and suspicious packages:

1. **`num_versions`**: Healthy packages tend to have a significantly higher number of published versions compared to suspicious packages. Suspicious packages often have very few versions (frequently just 1 or 2).
2. **`release_velocity`**: Healthy packages exhibit a more consistent release velocity, whereas suspicious packages typically have a very low velocity or sudden bursts.
3. **`stargazers_count`** (or `contributor_count`): Healthy packages generally have many more GitHub stars and contributors. Most suspicious packages lack a valid GitHub repository entirely, resulting in zero stars or missing GitHub stats.

These features show strong discriminative power and will be crucial for our machine learning models.

### Weakest Features
These features show massive overlap between classes, offering very little separation:

1. **`has_postinstall`**: Both healthy and suspicious packages may or may not use post-install scripts. While malicious packages often abuse postinstall scripts, benign ones use them just as frequently for legitimate build steps. The overlap is nearly total. **Decision**: *Drop or combine with other anomaly metrics (it might be useful when combined with low stargazers count, but alone it is weak).*
2. **`days_since_last_commit`**: Since many suspicious packages lack GitHub data entirely (defaulting to zero or max days), or some benign packages are mature and untouched for a long time, this feature has high overlap. **Decision**: *Consider dropping, or replacing with a binary `has_github_repo` flag (which captures the real separation).*
3. **`has_github_repo`**: While there is *some* difference, the overlap is substantial because many purely internal or small but healthy NPM packages do not explicitly link a GitHub repository. **Decision**: *Keep as a basic filter, but don't rely on it too heavily as a primary feature.*
